In [ ]:
import subprocess
import sys
import os

print("Installing Unsloth, TRL, PEFT, and bitsandbytes with internet enabled...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "torch", "torchvision", "torchaudio",
    "bitsandbytes", "transformers", "accelerate", "trl", "peft"
], check=True)

try:
    import unsloth
except ImportError:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/unslothai/unsloth.git"
    ], check=True)

print("Environment setup and Unsloth installation complete.")


In [ ]:
import base64
import json
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
DATA_DIR = WORKING_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

B64_TRAIN = "..."
B64_VAL = "..."

train_path = DATA_DIR / "train.jsonl"
train_path.write_bytes(base64.b64decode(B64_TRAIN))

val_path = DATA_DIR / "val.jsonl"
val_path.write_bytes(base64.b64decode(B64_VAL))

with open(train_path, "r", encoding="utf-8") as f:
    train_examples = [json.loads(line) for line in f if line.strip()]

with open(val_path, "r", encoding="utf-8") as f:
    val_examples = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(train_examples)} training rows and {len(val_examples)} held-out validation rows.")


In [ ]:
import torch
from unsloth import FastLanguageModel
from pathlib import Path

max_seq_length = 4096
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
load_in_4bit = True

print(f"Loading Gemma 31B in 4-bit QLoRA (max_seq_length={max_seq_length}, dtype={dtype})...")

LOCAL_MODEL_PATH = Path("/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2")
model_name = str(LOCAL_MODEL_PATH) if LOCAL_MODEL_PATH.exists() else "google/gemma-4-31b-it"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("Configuring anti-memorization Rank-8 LoRA adapter on attention and output heads (q_proj, v_proj, o_proj)...")
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()
print("Model initialized and PEFT LoRA configured successfully.")


In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

train_dataset = load_dataset("json", data_files=str(train_path), split="train")
val_dataset = load_dataset("json", data_files=str(val_path), split="train")
print(f"HuggingFace datasets loaded: train={len(train_dataset)}, val={len(val_dataset)}")

OUTPUT_DIR = WORKING_DIR / "training_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=3,
    max_steps=20,
    learning_rate=2.0e-5,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=2,
    eval_strategy="steps",
    eval_steps=5,
    optim="adamw_8bit",
    weight_decay=0.05,
    lr_scheduler_type="cosine",
    neftune_noise_alpha=5,
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print("Starting Unsloth LoRA fine-tuning with NEFTune noise alpha 5 and held-out validation...")
train_stats = trainer.train()
print("Training complete! Train stats:")
print(train_stats)


In [ ]:
from pathlib import Path

ADAPTER_DIR = WORKING_DIR / "adapters"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

print(f"Exporting LoRA adapter weights to {ADAPTER_DIR}...")
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

print("\n================ EXPORTED ADAPTER FILES ================")
total_size = 0
for p in sorted(ADAPTER_DIR.iterdir()):
    if p.is_file():
        sz = p.stat().st_size
        total_size += sz
        print(f"  - {p.name} ({sz / (1024 * 1024):.2f} MiB)")

print(f"Total Adapter Size: {total_size / (1024 * 1024):.2f} MiB")
print(f"SUCCESS: Ready to mount in my_submission/adapters/ for competition submission!")
